# Linked Data Demo (Fast)

Notebook court pour tester uniquement la federation RDF4J -> Wikidata et afficher une carte.


In [ ]:
import os
import re
import time
import requests
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path


In [ ]:
SPARQL_ENDPOINT = os.getenv('SPARQL_ENDPOINT')
SPARQL_USER = os.getenv('SPARQL_USER')
SPARQL_PASSWORD = os.getenv('SPARQL_PASSWORD')
SPARQL_TIMEOUT_SECONDS = int(os.getenv('SPARQL_TIMEOUT_SECONDS', '60'))
SPARQL_VERIFY_SSL = os.getenv('SPARQL_VERIFY_SSL', 'true').lower() == 'true'
USER_AGENT = os.getenv('USER_AGENT', 'Humanistica-Analytics/0.1')

missing = [k for k, v in {
    'SPARQL_ENDPOINT': SPARQL_ENDPOINT,
    'SPARQL_USER': SPARQL_USER,
    'SPARQL_PASSWORD': SPARQL_PASSWORD,
}.items() if not v]
if missing:
    raise RuntimeError(f"Variables d'environnement manquantes: {missing}")

print('Endpoint:', SPARQL_ENDPOINT)


In [ ]:
def sparql_select(query: str, timeout: int | None = None) -> pd.DataFrame:
    resp = requests.get(
        SPARQL_ENDPOINT,
        params={'query': query},
        headers={
            'Accept': 'application/sparql-results+json',
            'User-Agent': USER_AGENT,
        },
        auth=(SPARQL_USER, SPARQL_PASSWORD),
        timeout=timeout or SPARQL_TIMEOUT_SECONDS,
        verify=SPARQL_VERIFY_SSL,
    )
    resp.raise_for_status()
    payload = resp.json()
    rows = payload.get('results', {}).get('bindings', [])
    return pd.DataFrame([{k: v.get('value') for k, v in row.items()} for row in rows])


In [ ]:
# Detect main data graph automatically
q_graphs = '''
SELECT ?g (COUNT(?s) AS ?triples)
WHERE { GRAPH ?g { ?s ?p ?o } }
GROUP BY ?g
ORDER BY DESC(?triples)
'''

df_graphs = sparql_select(q_graphs)
if df_graphs.empty:
    raise RuntimeError('No named graphs found')

DATA_GRAPH = df_graphs.iloc[0]['g']
print('Using DATA_GRAPH =', DATA_GRAPH)


In [ ]:
cache_dir = Path("cache")
cache_dir.mkdir(exist_ok=True)
cache_file = cache_dir / "wikidata_birth_commune_coords_full.csv"

# Premier run: False pour forcer une extraction complete
use_cache = False
batch_size = 50
max_batches = 20   # 20*50 = 1000 communes candidates
sleep_seconds = 1.2

if use_cache and cache_file.exists():
    df_geo = pd.read_csv(cache_file)
    print("Loaded from cache:", cache_file, "rows =", len(df_geo))
else:
    all_parts = []

    for b in range(max_batches):
        offset = b * batch_size
        q = f"""
PREFIX core: <https://sdhss.org/ontology/core/>
PREFIX supp: <https://sdhss.org/ontology/crm-supplement/>
PREFIX sdh: <https://sdhss.org/ontology/shortcuts/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT ?commune ?name ?wd ?coord
WHERE {{
  {{
    SELECT ?commune ?name ?wd
    WHERE {{
      GRAPH <{DATA_GRAPH}> {{
        ?commune a core:C13 ; supp:P20 ?eid ; sdh:P9 ?name .
        FILTER(CONTAINS(STR(?commune), '/BirthCommune/'))
        ?eid owl:sameAs ?wd .
        FILTER(STRSTARTS(STR(?wd), 'http://www.wikidata.org/entity/'))
      }}
    }}
    ORDER BY ?commune
    LIMIT {batch_size}
    OFFSET {offset}
  }}
  SERVICE <https://query.wikidata.org/sparql> {{
    ?wd wdt:P625 ?coord .
  }}
}}
"""
        try:
            t0 = time.time()
            part = sparql_select(q, timeout=max(180, SPARQL_TIMEOUT_SECONDS))
            dt = round(time.time() - t0, 2)
            print(f"batch {b+1}/{max_batches} offset={offset} -> {len(part)} rows in {dt}s")
            if part.empty:
                break
            all_parts.append(part)
        except Exception as e:
            print(f"batch {b+1} failed: {type(e).__name__}: {e}")

        time.sleep(sleep_seconds)

    if all_parts:
        df_map = pd.concat(all_parts, ignore_index=True).drop_duplicates(subset=["commune", "wd"])
    else:
        df_map = pd.DataFrame(columns=["commune", "name", "wd", "coord"])

    def parse_wkt_point(wkt: str):
        m = re.match(r"^Point\(([-0-9.]+) ([-0-9.]+)\)$", str(wkt))
        if not m:
            return (np.nan, np.nan)
        return float(m.group(1)), float(m.group(2))

    if df_map.empty or "coord" not in df_map.columns:
        df_geo = pd.DataFrame(columns=["commune", "name", "wd", "coord", "lon", "lat"])
        print("No federated coordinate rows available.")
    else:
        coords = df_map["coord"].apply(parse_wkt_point)
        df_map["lon"] = coords.apply(lambda x: x[0])
        df_map["lat"] = coords.apply(lambda x: x[1])
        df_geo = df_map.dropna(subset=["lat", "lon"]).copy()
        print("Rows with valid coords:", len(df_geo))

    df_geo.to_csv(cache_file, index=False)
    print("Saved cache:", cache_file)

print(df_geo.head())

In [ ]:
if df_geo.empty:
    print('No map to show.')
else:
    fig = px.scatter_geo(
        df_geo,
        lat='lat',
        lon='lon',
        hover_name='name',
        hover_data={'wd': True, 'lat': ':.4f', 'lon': ':.4f'},
        title='Communes de naissance enrichies via Wikidata',
        opacity=0.6,
    )
    fig.update_traces(marker=dict(size=7, color='#1f77b4', line=dict(width=0.5, color='white')))
    fig.update_geos(
        scope='europe',
        projection_type='natural earth',
        showcountries=True,
        countrycolor='rgba(120,120,120,0.5)',
        showland=True,
        landcolor='rgb(242, 242, 242)',
        showocean=True,
        oceancolor='rgb(229, 244, 255)',
        lataxis_range=[40, 52],
        lonaxis_range=[-6, 10],
    )
    fig.update_layout(
        width=1250,
        height=780,
        margin=dict(l=20, r=20, t=70, b=20),
    )
    fig.show()

In [ ]:
print('Rows plotted:', len(df_geo))
if not df_geo.empty:
    unique_communes = df_geo['commune'].nunique()
    unique_wd = df_geo['wd'].nunique()
    print('Unique communes:', unique_communes)
    print('Unique Wikidata entities:', unique_wd)
